# CS781 Phase 3 — Rhetorical Role prediction with our Phase-1 models

**One run = all experiments** listed in the first cell; each writes its own `submission.zip`, plus `phase3_all_submissions.zip` with everything.

**Model:** sentences of a document are packed into 512-token windows (document context), the Phase-1 backbone encodes the window, each sentence is mean-pooled into one vector and a head predicts its rhetorical role. Overlapping windows are averaged. Training uses CL+IT train, model selection uses the official CL+IT dev sets (macro-F1).


In [ ]:
EXPERIMENTS = [
    dict(ARCH="hybrid_prefixlm",  METHOD="full",   LR=3e-5),
    dict(ARCH="hybrid_prefixlm",  METHOD="lorma",  LR=3e-4),
    dict(ARCH="decoder_llama",    METHOD="lora",   LR=5e-4),
    dict(ARCH="encdec_t5",        METHOD="rslora", LR=2e-4),
    dict(ARCH="efficient_retnet", METHOD="ia3",    LR=3e-3),
    dict(ARCH="decoder_llama",    METHOD="prompt", LR=3e-2),
    dict(ARCH="efficient_retnet", METHOD="kd", KD_TEACHER_RUN="hybrid_prefixlm_full"),
    dict(ARCH="hybrid_prefixlm",  METHOD="lorma",  LR=3e-4, CONTEXT_MODE="sentence", SENT_HEAD="linear", USE_POSITION=False,
         RUN_NAME="hybrid_prefixlm_lorma_sentonly"),
    dict(ARCH="ensemble", METHOD="ensemble", RUN_NAME="ensemble_top3", ENSEMBLE_RUNS="top3"),
]
TIME_BUDGET_HOURS = 11.0

CONTEXT_MODE = "doc"
SENT_HEAD = "bigru"
USE_POSITION = True
POS_BUCKETS = 10
LABEL_SMOOTHING = 0.1
ENSEMBLE_RUNS = []
CONTEXT = None
MAX_LEN = 512
MAX_SENT_TOKENS = 128
CHUNK_STRIDE_FRAC = 0.5

EPOCHS = 15
PATIENCE = 5
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
LR = None
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
GRAD_CLIP = 1.0
AMP = True
SEED = 42
LOG_EVERY = 50
CLASS_WEIGHTING = "inv_sqrt"

PARTIAL_TOP_K = 4
RANK, ALPHA, PEFT_DROPOUT = 16, 32, 0.05
PEFT_TARGET_FFN = True
ADAPTER_DIM = 64
N_PROMPT = 32
KD_TEACHER_RUN = None
KD_TEACHER_DIR = None
KD_KEEP_LAYERS = [0, 2, 4, 6, 8, 10]
KD_ALPHA, KD_TEMPERATURE = 0.5, 2.0
SAVE_WEIGHTS = False

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
DATA_PREFIX = "Phase_3_RR"
LOCAL_DATA_DIR = None
MODEL_DIR = None
SMOKE_TEST = False

DEFAULT_CONTEXT = {"decoder_llama": "bidir", "efficient_retnet": "echo", "hybrid_prefixlm": "bidir", "encdec_t5": "bidir"}
DEFAULT_LR = {"full": 3e-5, "kd": 1e-4, "partial": 1e-4, "lora": 5e-4, "lorma": 3e-4, "rslora": 2e-4, "ia3": 3e-3,
              "adapter": 1e-3, "bitfit": 1e-3, "prompt": 3e-2}
BASE_C = {k: v for k, v in dict(globals()).items() if k.isupper() and not k.startswith("_")}

def experiment_config(overrides):
    c = dict(BASE_C, **overrides)
    c["CONTEXT"] = overrides.get("CONTEXT") or DEFAULT_CONTEXT.get(c["ARCH"], "-")
    c["LR"] = overrides.get("LR") or DEFAULT_LR.get(c["METHOD"], 0.0)
    c["RUN_NAME"] = overrides.get("RUN_NAME") or f"{c['ARCH']}_{c['METHOD']}"
    if c["CONTEXT"] == "echo": assert c["MAX_LEN"] <= 640
    return c

C = experiment_config(EXPERIMENTS[0])
for _i, _e in enumerate(EXPERIMENTS, 1):
    _c = experiment_config(_e)
    print(f"{_i}. {_c['RUN_NAME']:34s} arch={_c['ARCH']:16s} method={_c['METHOD']:7s} ctx={_c['CONTEXT_MODE']:8s} lr={_c['LR']}")

In [ ]:
import os, sys, json, glob, time, random, shutil, zipfile
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()

def hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    for name in ("HF_TOKEN", "HF"):
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret(name)
            if tok: return tok.strip()
        except Exception:
            pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

HF_TOKEN_VALUE = hf_token()
if HF_TOKEN_VALUE:
    os.environ["HF_TOKEN"] = HF_TOKEN_VALUE
    print("Hugging Face token found (length", len(HF_TOKEN_VALUE), ")")
else:
    print("WARNING: no Hugging Face token found. On Kaggle: Add-ons -> Secrets -> add 'HF_TOKEN' (or 'HF') "
          "and switch its toggle ON for this notebook; the course dataset is gated.")

def data_file(rel):
    roots = [LOCAL_DATA_DIR] if LOCAL_DATA_DIR else []
    for root in roots:
        p = os.path.join(root, rel)
        if os.path.exists(p): return p
    hits = glob.glob(f"/kaggle/input/**/{rel}", recursive=True)
    if hits: return hits[0]
    from huggingface_hub import hf_hub_download
    assert HF_TOKEN_VALUE, ("No Hugging Face token: add the Kaggle secret HF_TOKEN (or HF), switch it ON for this notebook "
                           "(Add-ons -> Secrets), then run again.")
    return hf_hub_download(HF_DATASET_REPO, f"{DATA_PREFIX}/{rel}", repo_type="dataset", token=HF_TOKEN_VALUE)

def find_model_dir(arch):
    for cfgp in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        try:
            if json.load(open(cfgp)).get("arch") == arch and os.path.exists(os.path.join(os.path.dirname(cfgp), "pytorch_model.bin")):
                return os.path.dirname(cfgp)
        except Exception:
            pass
    raise FileNotFoundError(f"no Phase-1 model folder for arch={arch} under /kaggle/input "
                            "(attach the dataset that contains config.json + pytorch_model.bin + legal_bpe_tokenizer.json)")

### Phase-1 code (unchanged): tokenizer and the four architectures

In [ ]:
import heapq
import json
import os
import re
import time
from collections import Counter, defaultdict

PRETOKENIZE_PATTERN = (
    r"'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+"
)
EOS_TOKEN = "<|endoftext|>"

class LegalBPETokenizer:
    def __init__(self, merges=None, name="legal-bpe"):
        self.name = name
        self.merges = [tuple(m) for m in (merges or [])]
        self._pattern = re.compile(PRETOKENIZE_PATTERN)
        self._build_tables()

    def _build_tables(self):
        self.ranks = {pair: i for i, pair in enumerate(self.merges)}
        self.vocab = {i: bytes([i]) for i in range(256)}
        for i, (a, b) in enumerate(self.merges):
            self.vocab[256 + i] = self.vocab[a] + self.vocab[b]
        self.eos_token = EOS_TOKEN
        self.eos_token_id = 256 + len(self.merges)
        self.pad_token = self.eos_token
        self.pad_token_id = self.eos_token_id
        self.model_max_length = 10**9
        self._cache = {}

    def __len__(self):
        return 256 + len(self.merges) + 1

    @property
    def vocab_size(self):
        return len(self)

    def pretokenize(self, text):
        return self._pattern.findall(text)

    @classmethod
    def train(cls, texts, vocab_size, name="legal-bpe", min_frequency=2, verbose=True):
        assert 258 <= vocab_size <= 65535, "vocab_size must fit in uint16"
        t0 = time.time()
        pattern = re.compile(PRETOKENIZE_PATTERN)

        word_counts = Counter()
        n_chars = 0
        for text in texts:
            n_chars += len(text)
            word_counts.update(pattern.findall(text))
        if verbose:
            print(f"[BPE] {n_chars:,} characters, {len(word_counts):,} unique words "
                  f"({time.time() - t0:.0f}s)")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_counts = defaultdict(int)
        pair_where = defaultdict(set)
        for wi, (w, f) in enumerate(zip(words, freqs)):
            for a, b in zip(w, w[1:]):
                pair_counts[(a, b)] += f
                pair_where[(a, b)].add(wi)
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        n_merges = vocab_size - 257
        merges = []
        while len(merges) < n_merges and heap:
            neg_c, pair = heapq.heappop(heap)
            count = pair_counts.get(pair, 0)
            if count <= 0:
                continue
            if -neg_c != count:
                heapq.heappush(heap, (-count, pair))
                continue
            if count < min_frequency:
                if verbose:
                    print(f"[BPE] stopping early: best pair frequency {count} < {min_frequency}")
                break

            new_id = 256 + len(merges)
            merges.append(pair)
            a, b = pair
            changed = set()
            for wi in list(pair_where[pair]):
                w, f = words[wi], freqs[wi]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                new_w, i = [], 0
                while i < len(w):
                    if i + 1 < len(w) and w[i] == a and w[i + 1] == b:
                        new_w.append(new_id)
                        i += 2
                    else:
                        new_w.append(w[i])
                        i += 1
                words[wi] = new_w
                for p in zip(new_w, new_w[1:]):
                    pair_counts[p] += f
                    pair_where[p].add(wi)
                    changed.add(p)
            del pair_where[pair]
            pair_counts.pop(pair, None)
            changed.discard(pair)
            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_where.pop(p, None)
            if verbose and (len(merges) % 2000 == 0):
                print(f"[BPE] {len(merges):,}/{n_merges:,} merges ({time.time() - t0:.0f}s)")

        tok = cls(merges, name=name)
        if verbose:
            print(f"[BPE] done: vocab size {len(tok):,} ({time.time() - t0:.0f}s)")
        return tok

    def _encode_word(self, word):
        ids = self._cache.get(word)
        if ids is not None:
            return ids
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) >= 2:
            best_rank, best_pair = None, None
            for p in zip(ids, ids[1:]):
                r = ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_pair = r, p
            if best_pair is None:
                break
            a, b = best_pair
            new_id = 256 + best_rank
            merged, i = [], 0
            while i < len(ids):
                if i + 1 < len(ids) and ids[i] == a and ids[i + 1] == b:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(ids[i])
                    i += 1
            ids = merged
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = ids
        return ids

    def encode(self, text, add_special_tokens=False):
        out = []
        for word in self._pattern.findall(text):
            out.extend(self._encode_word(word))
        return out

    def __call__(self, texts, add_special_tokens=False, **kwargs):
        if isinstance(texts, str):
            return {"input_ids": self.encode(texts)}
        return {"input_ids": [self.encode(t) for t in texts]}

    def decode(self, ids, skip_special_tokens=True, **kwargs):
        parts = []
        for i in ids:
            i = int(i)
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(self.eos_token.encode("utf-8"))
                continue
            b = self.vocab.get(i)
            if b is not None:
                parts.append(b)
        return b"".join(parts).decode("utf-8", errors="replace")

    def batch_decode(self, sequences, skip_special_tokens=True, **kwargs):
        return [self.decode(s, skip_special_tokens=skip_special_tokens) for s in sequences]

    def convert_ids_to_tokens(self, ids):
        return [self.vocab[i].decode("utf-8", errors="replace") if i in self.vocab else self.eos_token
                for i in ids]

    def save_pretrained(self, directory):
        os.makedirs(directory, exist_ok=True)
        path = os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path, "w") as f:
            json.dump({
                "type": "byte-level-bpe",
                "name": self.name,
                "pattern": PRETOKENIZE_PATTERN,
                "eos_token": self.eos_token,
                "eos_token_id": self.eos_token_id,
                "vocab_size": len(self),
                "merges": [list(m) for m in self.merges],
            }, f)
        return path

    @classmethod
    def from_pretrained(cls, directory):
        path = directory if directory.endswith(".json") else os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path) as f:
            data = json.load(f)
        assert data["pattern"] == PRETOKENIZE_PATTERN, "tokenizer was saved with a different pre-tokenisation pattern"
        tok = cls(data["merges"], name=data.get("name", "legal-bpe"))
        assert tok.eos_token_id == data["eos_token_id"]
        return tok

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_positions, theta=10_000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", torch.cat([freqs.cos(), freqs.cos()], dim=-1), persistent=False)
        self.register_buffer("sin_cached", torch.cat([freqs.sin(), freqs.sin()], dim=-1), persistent=False)

    def forward(self, x, offset=0):
        T = x.size(-2)
        cos = self.cos_cached[offset:offset + T].to(x.dtype)
        sin = self.sin_cached[offset:offset + T].to(x.dtype)
        x1, x2 = x.chunk(2, dim=-1)
        return x * cos + torch.cat((-x2, x1), dim=-1) * sin

class SelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout

    def forward(self, x, rope, attn_mask=None, is_causal=True, cache=None, offset=0):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k = rope(q, offset), rope(k, offset)
        if cache is not None:
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=is_causal and attn_mask is None,
            dropout_p=self.dropout if self.training else 0.0,
        )
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, hidden):
        super().__init__()
        self.w_gate_up = nn.Linear(n_embd, 2 * hidden, bias=False)
        self.w_down = nn.Linear(hidden, n_embd, bias=False)

    def forward(self, x):
        gate, up = self.w_gate_up(x).chunk(2, dim=-1)
        return self.w_down(F.silu(gate) * up)

class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope, attn_mask=None, is_causal=True, cache=None, offset=0):
        x = x + self.drop(self.attn(self.norm1(x), rope, attn_mask, is_causal, cache, offset))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

def init_weights(model, n_layers, std=0.02):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
    for name, p in model.named_parameters():
        if name.endswith("proj.weight") or name.endswith("w_down.weight"):
            nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * n_layers))

def lm_loss(lm_head, hidden, input_ids, loss_start=0):
    start = max(int(loss_start), 1)
    logits = lm_head(hidden[:, start - 1:-1])
    targets = input_ids[:, start:]
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

class DecoderOnlyLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList(
            [DecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)]
        )
        self.norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, caches=None, offset=0, is_causal=True):
        x = self.drop(self.embed(input_ids))
        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, is_causal=is_causal,
                      cache=None if caches is None else caches[i], offset=offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len=0):
        return lm_loss(self.lm_head, self._hidden(input_ids), input_ids, prefix_len)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        caches = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, caches, offset=0, is_causal=True)
        return self.lm_head(h[:, -1]), {"caches": caches, "pos": prompt_ids.size(1)}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["caches"], offset=state["pos"], is_causal=False)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        ids = torch.cat([prompt_ids, cont_ids], dim=1)
        h = self._hidden(ids)
        return self.lm_head(h[:, prompt_ids.size(1) - 1:-1])

def prefix_lm_mask(T, prefix_len, device):
    i = torch.arange(T, device=device)
    mask = (i[None, :] <= i[:, None]) | (i[None, :] < prefix_len)
    return mask[None, None]

class PrefixLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.segment = nn.Embedding(2, d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList(
            [DecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)]
        )
        self.norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, prefix_len, caches=None, offset=0, decoding=False):
        B, T = input_ids.shape
        positions = torch.arange(offset, offset + T, device=input_ids.device)
        segments = (positions >= prefix_len).long()
        x = self.drop(self.embed(input_ids) + self.segment(segments)[None])

        if decoding:
            attn_mask, is_causal = None, False
        elif prefix_len >= T:
            attn_mask, is_causal = None, False
        elif prefix_len <= 0:
            attn_mask, is_causal = None, True
        else:
            attn_mask, is_causal = prefix_lm_mask(T, prefix_len, input_ids.device), False

        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, attn_mask=attn_mask, is_causal=is_causal,
                      cache=None if caches is None else caches[i], offset=offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len):
        P = int(prefix_len)
        return lm_loss(self.lm_head, self._hidden(input_ids, P), input_ids, P)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        P = prompt_ids.size(1)
        caches = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, prefix_len=P, caches=caches, offset=0)
        return self.lm_head(h[:, -1]), {"caches": caches, "pos": P, "prefix_len": P}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["prefix_len"], state["caches"],
                         offset=state["pos"], decoding=True)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        P = prompt_ids.size(1)
        h = self._hidden(torch.cat([prompt_ids, cont_ids], dim=1), prefix_len=P)
        return self.lm_head(h[:, P - 1:-1])

class CrossAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.0):
        super().__init__()
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.q = nn.Linear(n_embd, n_embd, bias=False)
        self.kv = nn.Linear(n_embd, 2 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout

    def forward(self, x, memory, cache=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if cache is not None and "k" in cache:
            k, v = cache["k"], cache["v"]
        else:
            S = memory.size(1)
            k, v = self.kv(memory).view(B, S, 2, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
            if cache is not None:
                cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class EncoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope):
        x = x + self.drop(self.attn(self.norm1(x), rope, is_causal=False))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

class CrossDecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.self_attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.cross_attn = CrossAttention(n_embd, n_head, dropout)
        self.norm3 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, rope, cache=None, offset=0, is_causal=True):
        self_cache = None if cache is None else cache["self"]
        cross_cache = None if cache is None else cache["cross"]
        x = x + self.drop(self.self_attn(self.norm1(x), rope, is_causal=is_causal, cache=self_cache, offset=offset))
        x = x + self.drop(self.cross_attn(self.norm2(x), memory, cross_cache))
        x = x + self.drop(self.mlp(self.norm3(x)))
        return x

class EncoderDecoderLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, H = cfg["n_embd"], cfg["n_head"]
        self.bos_id = cfg["bos_token_id"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.enc_blocks = nn.ModuleList(
            [EncoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(cfg["n_enc_layer"])]
        )
        self.enc_norm = RMSNorm(d)
        self.dec_blocks = nn.ModuleList(
            [CrossDecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(cfg["n_dec_layer"])]
        )
        self.dec_norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, cfg["n_enc_layer"] + cfg["n_dec_layer"])

    def encode(self, src_ids):
        x = self.drop(self.embed(src_ids))
        for block in self.enc_blocks:
            x = block(x, self.rope)
        return self.enc_norm(x)

    def decode(self, tgt_in, memory, caches=None, offset=0, is_causal=True):
        x = self.drop(self.embed(tgt_in))
        for i, block in enumerate(self.dec_blocks):
            x = block(x, memory, self.rope, None if caches is None else caches[i], offset, is_causal)
        return self.dec_norm(x)

    def _shift_right(self, tgt):
        bos = torch.full_like(tgt[:, :1], self.bos_id)
        return torch.cat([bos, tgt[:, :-1]], dim=1)

    def forward(self, input_ids, prefix_len):
        P = int(prefix_len)
        assert 0 < P < input_ids.size(1), "encoder-decoder needs 0 < prefix_len < block size"
        src, tgt = input_ids[:, :P], input_ids[:, P:]
        h = self.decode(self._shift_right(tgt), self.encode(src))
        logits = self.lm_head(h)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))

    @torch.no_grad()
    def prefill(self, prompt_ids):
        memory = self.encode(prompt_ids)
        caches = [{"self": {}, "cross": {}} for _ in self.dec_blocks]
        bos = torch.full((prompt_ids.size(0), 1), self.bos_id, dtype=torch.long, device=prompt_ids.device)
        h = self.decode(bos, memory, caches, offset=0, is_causal=True)
        return self.lm_head(h[:, -1]), {"memory": memory, "caches": caches, "pos": 1}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self.decode(token_ids[:, None], state["memory"], state["caches"], offset=state["pos"], is_causal=False)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        return self.lm_head(self.decode(self._shift_right(cont_ids), self.encode(prompt_ids)))

class MultiScaleRetention(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkvg = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        gammas = 1.0 - 2.0 ** (-5.0 - torch.arange(n_head, dtype=torch.float32))
        self.register_buffer("log_gamma", torch.log(gammas), persistent=False)

    def _decay_matrix(self, T, device):
        idx = torch.arange(T, device=device, dtype=torch.float32)
        diff = idx[:, None] - idx[None, :]
        decay = torch.exp(diff.clamp(min=0)[None] * self.log_gamma[:, None, None])
        decay = decay.masked_fill(diff[None] < 0, 0.0)
        return decay / decay.sum(dim=-1, keepdim=True).sqrt()

    def forward(self, x, rope, state=None, offset=0):
        B, T, C = x.shape
        q, k, v, g = self.qkvg(x).split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = rope(q, offset), rope(k, offset)
        k = k * self.head_dim ** -0.5

        if state is not None and "S" in state:
            with torch.autocast(device_type=x.device.type, enabled=False):
                gamma = self.log_gamma.exp().view(1, -1, 1, 1)
                state["S"] = gamma * state["S"] + k.float().transpose(-1, -2) @ v.float()
                state["scale"] = gamma * state["scale"] + 1.0
                y = ((q.float() @ state["S"]) / state["scale"].sqrt()).to(v.dtype)
        else:
            decay = self._decay_matrix(T, x.device).to(q.dtype)
            y = ((q @ k.transpose(-1, -2)) * decay) @ v
            if state is not None:
                with torch.autocast(device_type=x.device.type, enabled=False):
                    pos = torch.arange(T, device=x.device, dtype=torch.float32)
                    w = torch.exp((T - 1 - pos)[None, :] * self.log_gamma[:, None])
                    state["S"] = torch.einsum("bhtd,ht,bhte->bhde", k.float(), w, v.float())
                    state["scale"] = w.sum(dim=-1).view(1, -1, 1, 1)

        y = y.transpose(1, 2).float()
        y = y * torch.rsqrt(y.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
        y = y.reshape(B, T, C).to(g.dtype)
        return self.proj(F.silu(g) * y)

class FeedForward(nn.Module):
    def __init__(self, n_embd, hidden):
        super().__init__()
        self.w_in = nn.Linear(n_embd, hidden)
        self.w_down = nn.Linear(hidden, n_embd)

    def forward(self, x):
        return self.w_down(F.gelu(self.w_in(x)))

class RetNetBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.retention = MultiScaleRetention(n_embd, n_head)
        self.norm2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope, state=None, offset=0):
        x = x + self.drop(self.retention(self.norm1(x), rope, state, offset))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x

class RetNetLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList([RetNetBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)])
        self.norm = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, states=None, offset=0):
        x = self.drop(self.embed(input_ids))
        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, None if states is None else states[i], offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len=0):
        return lm_loss(self.lm_head, self._hidden(input_ids), input_ids, prefix_len)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        states = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, states, offset=0)
        return self.lm_head(h[:, -1]), {"states": states, "pos": prompt_ids.size(1)}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["states"], offset=state["pos"])
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        ids = torch.cat([prompt_ids, cont_ids], dim=1)
        return self.lm_head(self._hidden(ids)[:, prompt_ids.size(1) - 1:-1])

ARCH_CLASSES = {'decoder_llama': DecoderOnlyLM, 'hybrid_prefixlm': PrefixLM, 'encdec_t5': EncoderDecoderLM, 'efficient_retnet': RetNetLM}

### Backbone wrapper and fine-tuning methods (LoRA, LoRMA, rsLoRA, IA3, prompt tuning, KD, ...)

In [ ]:
import math, time, json, os
import torch
import torch.nn as nn
import torch.nn.functional as F

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale

class RSLoRALinear(LoRALinear):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__(base, r, alpha, dropout)
        self.scale = alpha / math.sqrt(r)

class IA3Linear(nn.Module):
    def __init__(self, base, mode, n_split=3):
        super().__init__()
        self.base, self.mode = base, mode
        if mode == "out_kv":
            d = base.out_features // n_split
            self.d = d
            self.l_kv = nn.Parameter(torch.ones(2 * d))
        else:
            self.l_in = nn.Parameter(torch.ones(base.in_features))
    def forward(self, x):
        if self.mode == "in":
            return self.base(x * self.l_in)
        y = self.base(x)
        d = self.d
        scale = torch.cat([torch.ones(d, device=y.device, dtype=y.dtype), self.l_kv.to(y.dtype),
                           torch.ones(y.size(-1) - 3 * d, device=y.device, dtype=y.dtype)])
        return y * scale

class LoRMALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        y = self.base(x)
        return y + (self.drop(y) @ self.A.t() @ self.B.t()) * self.scale

class AdapterLinear(nn.Module):
    def __init__(self, base, bottleneck=64):
        super().__init__()
        self.base = base
        self.down = nn.Linear(base.out_features, bottleneck)
        self.up = nn.Linear(bottleneck, base.out_features)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))

class BiasLinear(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.bias = nn.Parameter(torch.zeros(base.out_features))
    def forward(self, x):
        return self.base(x) + self.bias

ATTN_LINEARS = ("qkv", "proj", "qkvg", "q", "kv")
FFN_LINEARS = ("w_gate_up", "w_down", "w_in")
OUT_PROJ = ("proj", "w_down")

def _replace_linears(module, names, make):
    n = 0
    for parent in list(module.modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names:
                setattr(parent, cname, make(child)); n += 1
    return n

def backbone_blocks(arch, bb):
    return list(bb.enc_blocks) if arch == "encdec_t5" else list(bb.blocks)

class NERModel(nn.Module):
    def __init__(self, arch, backbone, n_tags, context="bidir", head="linear", n_prompt=0, dropout=0.1):
        super().__init__()
        self.arch, self.bb, self.context = arch, backbone, context
        self.head_out = n_tags
        d = backbone.cfg["n_embd"]
        self.drop = nn.Dropout(dropout)
        if head == "mlp":
            self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, n_tags))
        else:
            self.head = nn.Linear(d, n_tags)
        self.prompt = nn.Parameter(torch.randn(n_prompt, d) * 0.02) if n_prompt > 0 else None
        if hasattr(backbone, "lm_head"):
            backbone.lm_head = nn.Identity()
        if arch == "encdec_t5":
            backbone.dec_blocks = nn.ModuleList(); backbone.dec_norm = nn.Identity()

    def _mask(self, key_ok, T, causal):
        m = key_ok[:, None, None, :].expand(-1, 1, T, T)
        if causal:
            m = m & torch.ones(T, T, dtype=torch.bool, device=key_ok.device).tril()[None, None]
        eye = torch.eye(T, dtype=torch.bool, device=key_ok.device)[None, None]
        return m | eye

    def hidden(self, ids, attn):
        bb, arch = self.bb, self.arch
        B, T = ids.shape
        x = bb.embed(ids)
        ok = attn.bool()
        if self.context == "echo" and arch in ("decoder_llama", "efficient_retnet"):
            x = torch.cat([x, x], dim=1); ok = torch.cat([ok, ok], dim=1)
        if arch == "hybrid_prefixlm":
            x = x + bb.segment(torch.zeros(x.size(1), dtype=torch.long, device=x.device))[None]
        P = 0
        if self.prompt is not None:
            P = self.prompt.size(0)
            x = torch.cat([self.prompt[None].expand(B, -1, -1).to(x.dtype), x], dim=1)
            ok = torch.cat([torch.ones(B, P, dtype=torch.bool, device=x.device), ok], dim=1)
        x = bb.drop(x)
        L = x.size(1)
        if arch == "efficient_retnet":
            for blk in bb.blocks:
                x = blk(x, bb.rope)
            x = bb.norm(x)
        elif arch == "encdec_t5":
            mask = self._mask(ok, L, causal=False)
            for blk in bb.enc_blocks:
                x = x + blk.drop(blk.attn(blk.norm1(x), bb.rope, attn_mask=mask, is_causal=False))
                x = x + blk.drop(blk.mlp(blk.norm2(x)))
            x = bb.enc_norm(x)
        else:
            causal = self.context in ("causal", "echo") and arch == "decoder_llama"
            mask = self._mask(ok, L, causal=causal)
            for blk in bb.blocks:
                x = blk(x, bb.rope, attn_mask=mask, is_causal=False)
            x = bb.norm(x)
        x = x[:, P:]
        if self.context == "echo" and arch in ("decoder_llama", "efficient_retnet"):
            x = x[:, T:]
        return x

    def forward(self, ids, attn):
        return self.head(self.drop(self.hidden(ids, attn)))

def load_backbone(model_dir, arch_classes, map_location="cpu"):
    cfg = json.load(open(os.path.join(model_dir, "config.json")))
    bb = arch_classes[cfg["arch"]](cfg)
    sd = torch.load(os.path.join(model_dir, "pytorch_model.bin"), map_location=map_location)
    missing, unexpected = bb.load_state_dict(sd, strict=False)
    missing = [k for k in missing if not k.endswith("lm_head.weight")]
    assert not missing and not unexpected, (missing, unexpected)
    return cfg, bb

def keep_layers(arch, bb, keep):
    if arch == "encdec_t5":
        bb.enc_blocks = nn.ModuleList([bb.enc_blocks[i] for i in keep]); bb.cfg["n_enc_layer"] = len(keep)
    else:
        bb.blocks = nn.ModuleList([bb.blocks[i] for i in keep]); bb.cfg["n_layer"] = len(keep)
    return bb

def apply_method(model, method, C):
    arch, bb = model.arch, model.bb
    info = {}
    if method in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if method == "partial":
            blocks = backbone_blocks(arch, bb)
            for blk in blocks[-C["PARTIAL_TOP_K"]:]:
                for p in blk.parameters(): p.requires_grad = True
            norm = bb.enc_norm if arch == "encdec_t5" else bb.norm
            for p in norm.parameters(): p.requires_grad = True
        elif method in ("lora", "lorma", "rslora"):
            cls = {"lora": LoRALinear, "lorma": LoRMALinear, "rslora": RSLoRALinear}[method]
            targets = ATTN_LINEARS + (FFN_LINEARS if C["PEFT_TARGET_FFN"] else ())
            info["wrapped"] = _replace_linears(bb, targets, lambda l: cls(l, C["RANK"], C["ALPHA"], C["PEFT_DROPOUT"]))
        elif method == "adapter":
            info["wrapped"] = _replace_linears(bb, OUT_PROJ, lambda l: AdapterLinear(l, C["ADAPTER_DIM"]))
        elif method == "bitfit":
            info["wrapped"] = _replace_linears(bb, ATTN_LINEARS + FFN_LINEARS, BiasLinear)
            for n, p in bb.named_parameters():
                if n.endswith("bias") or "norm" in n: p.requires_grad = True
        elif method == "ia3":
            n = _replace_linears(bb, ("qkv",), lambda l: IA3Linear(l, "out_kv", 3))
            n += _replace_linears(bb, ("qkvg",), lambda l: IA3Linear(l, "out_kv", 4))
            n += _replace_linears(bb, ("w_down",), lambda l: IA3Linear(l, "in"))
            info["wrapped"] = n
        elif method == "prompt":
            pass
        else:
            raise ValueError(method)
        for p in model.head.parameters(): p.requires_grad = True
        if model.prompt is not None: model.prompt.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    info.update(trainable_params=n_train, total_params=n_total, trainable_pct=100.0 * n_train / n_total)
    return info

### Rhetorical-role data, sentence model, training and metrics

In [ ]:
import json, os, math, time, random
from collections import Counter, defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def parse_docs(obj, domain):
    items = obj.items() if isinstance(obj, dict) and "text" not in obj else (
        [(None, obj)] if isinstance(obj, dict) else [(None, d) for d in obj])
    docs = []
    for k, d in items:
        if isinstance(d, list):
            d = {"text": d}
        did = str(d.get("id", d.get("doc_id", k)))
        sents = d.get("text", d.get("sentences"))
        if isinstance(sents, str):
            sents = [sents]
        labs = d.get("labels", d.get("label"))
        docs.append({"id": did, "domain": domain, "sents": [str(s) for s in sents],
                     "labels": [str(l) for l in labs] if labs is not None else None})
    return docs

def load_label_vocab(obj):
    if isinstance(obj, dict):
        if "labels" in obj: return list(obj["labels"])
        if all(isinstance(v, int) for v in obj.values()):
            return [k for k, _ in sorted(obj.items(), key=lambda x: x[1])]
        return list(obj.keys())
    return list(obj)

def check_docs(docs, labels, name):
    st = Counter()
    for d in docs:
        st["docs"] += 1; st["sentences"] += len(d["sents"])
        if d["labels"] is not None:
            if len(d["labels"]) != len(d["sents"]): st["length_mismatch"] += 1
            st["unknown_label"] += sum(l not in labels for l in d["labels"])
        st["empty_sentences"] += sum(not s.strip() for s in d["sents"])
    print(f"{name}: {dict(st)}")
    return st

def encode_docs(docs, tok, C):
    for d in docs:
        d["ids"] = []
        for s in d["sents"]:
            ids = tok.encode(s)[: C["MAX_SENT_TOKENS"]]
            d["ids"].append(ids if ids else [tok.eos_token_id])
    return docs

def make_chunks(doc, C, eos):
    n = len(doc["ids"])
    if C["CONTEXT_MODE"] == "sentence":
        return [[i] for i in range(n)]
    chunks, start = [], 0
    while start < n:
        cur, length = [], 0
        i = start
        while i < n and (length + len(doc["ids"][i]) + 1 <= C["MAX_LEN"] or not cur):
            cur.append(i); length += len(doc["ids"][i]) + 1; i += 1
        chunks.append(cur)
        if i >= n: break
        start = cur[0] + max(1, int(len(cur) * C["CHUNK_STRIDE_FRAC"]))
    return chunks

def build_examples(docs, C, eos, label2id, with_labels=True):
    ex = []
    for di, d in enumerate(docs):
        for ch in make_chunks(d, C, eos):
            ids, seg = [], []
            for k, si in enumerate(ch):
                ids += d["ids"][si]; seg += [k] * len(d["ids"][si])
                ids.append(eos); seg.append(-1)
            ids, seg = ids[: C["MAX_LEN"]], seg[: C["MAX_LEN"]]
            labs = [label2id[d["labels"][si]] for si in ch] if (with_labels and d["labels"] is not None) else None
            nb = C["POS_BUCKETS"]
            pos = [min(nb - 1, int(nb * si / max(1, len(d["ids"])))) for si in ch]
            ex.append({"doc": di, "sents": ch, "ids": ids, "seg": seg, "labels": labs, "pos": pos})
    return ex

def collate(batch, pad_id, device):
    T = max(len(b["ids"]) for b in batch); S = max(len(b["sents"]) for b in batch)
    ids = torch.full((len(batch), T), pad_id, dtype=torch.long)
    att = torch.zeros((len(batch), T), dtype=torch.long)
    seg = torch.full((len(batch), T), -1, dtype=torch.long)
    lab = torch.full((len(batch), S), -100, dtype=torch.long)
    pos = torch.zeros((len(batch), S), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["ids"])
        ids[i, :n] = torch.tensor(b["ids"]); att[i, :n] = 1; seg[i, :n] = torch.tensor(b["seg"])
        if b["labels"] is not None:
            lab[i, :len(b["labels"])] = torch.tensor(b["labels"])
        pos[i, :len(b["pos"])] = torch.tensor(b["pos"])
    return ids.to(device), att.to(device), seg.to(device), lab.to(device), S, pos.to(device)

class BiGRUHead(nn.Module):
    def __init__(self, d, n_labels):
        super().__init__()
        self.gru = nn.GRU(d, d // 2, batch_first=True, bidirectional=True)
        self.out = nn.Linear(d, n_labels)
    def forward(self, x):
        return self.out(self.gru(x.float())[0].to(x.dtype))

class RRModel(NERModel):
    def __init__(self, arch, backbone, n_labels, context, sent_head="linear", n_prompt=0, dropout=0.1, pos_buckets=0):
        super().__init__(arch, backbone, n_labels, context=context, head="linear", n_prompt=n_prompt, dropout=dropout)
        d = backbone.cfg["n_embd"]
        self.head = BiGRUHead(d, n_labels) if sent_head == "bigru" else nn.Linear(d, n_labels)
        self.head_out = n_labels
        self.pos_emb = nn.Embedding(pos_buckets, d) if pos_buckets > 0 else None
        if self.pos_emb is not None: nn.init.zeros_(self.pos_emb.weight)

    def forward(self, ids, att, seg, n_sent, pos=None):
        h = self.hidden(ids, att)
        onehot = (seg.unsqueeze(-1) == torch.arange(n_sent, device=seg.device)).to(h.dtype)
        cnt = onehot.sum(1).clamp(min=1).unsqueeze(-1)
        sent = torch.einsum("bts,btd->bsd", onehot, h) / cnt
        if self.pos_emb is not None and pos is not None:
            sent = sent + self.pos_emb(pos).to(sent.dtype)
        return self.head(self.drop(sent))

def macro_scores(gold, pred, labels):
    tp, fp, fn = Counter(), Counter(), Counter()
    for g, p in zip(gold, pred):
        if g == p: tp[g] += 1
        else: fp[p] += 1; fn[g] += 1
    per = {}
    for l in labels:
        if tp[l] + fp[l] + fn[l] == 0: continue
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = {"P": P, "R": R, "F1": 2 * P * R / (P + R) if P + R else 0.0, "support": tp[l] + fn[l]}
    k = max(1, len(per))
    out = {m: sum(v[m] for v in per.values()) / k for m in ("P", "R", "F1")}
    out = {"macro_F1": out["F1"], "macro_P": out["P"], "macro_R": out["R"],
           "accuracy": sum(tp.values()) / max(1, len(gold)), "per_label": per}
    return out

def confusions(gold, pred, top=12):
    c = Counter((g, p) for g, p in zip(gold, pred) if g != p)
    return {f"{g}->{p}": n for (g, p), n in c.most_common(top)}

@torch.no_grad()
def predict_logits(model, docs, C, pad_id, device, label2id=None):
    model.eval()
    ex = build_examples(docs, C, pad_id, {}, with_labels=False)
    acc = [torch.zeros(len(d["sents"]), model.head_out) for d in docs]
    cnt = [torch.zeros(len(d["sents"]), 1) for d in docs]
    bs = C["EVAL_BATCH_SIZE"]
    for i in range(0, len(ex), bs):
        batch = ex[i:i + bs]
        ids, att, seg, _, S, pos = collate(batch, pad_id, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=C["AMP"] and device.type == "cuda"):
            lg = model(ids, att, seg, S, pos).float().cpu()
        for j, b in enumerate(batch):
            for k, si in enumerate(b["sents"]):
                acc[b["doc"]][si] += lg[j, k]; cnt[b["doc"]][si] += 1
    return [a / c.clamp(min=1) for a, c in zip(acc, cnt)]

def evaluate_docs(logits, docs, id2label, labels):
    gold, pred, dom = [], [], []
    for lg, d in zip(logits, docs):
        p = [id2label[int(i)] for i in lg.argmax(-1)]
        gold += d["labels"]; pred += p; dom += [d["domain"]] * len(p)
    res = macro_scores(gold, pred, labels)
    for D in sorted(set(dom)):
        idx = [i for i, x in enumerate(dom) if x == D]
        res[f"macro_F1_{D}"] = macro_scores([gold[i] for i in idx], [pred[i] for i in idx], labels)["macro_F1"]
    res["confusions"] = confusions(gold, pred)
    return res

def class_weights(examples, n, C, device):
    w = torch.ones(n)
    if C["CLASS_WEIGHTING"] == "inv_sqrt":
        cnt = torch.ones(n)
        for e in examples:
            for l in e["labels"]: cnt[l] += 1
        w = (cnt.sum() / cnt).sqrt(); w = w / w.mean()
    return w.to(device)

def train(model, train_ex, dev_docs, C, pad_id, device, id2label, labels, teacher=None, log=print):
    params = [p for p in model.parameters() if p.requires_grad]
    head_ids = {id(p) for p in model.head.parameters()}
    if model.pos_emb is not None: head_ids |= {id(p) for p in model.pos_emb.parameters()}
    if model.prompt is not None: head_ids.add(id(model.prompt))
    groups = [g for g in ({"params": [p for p in params if id(p) in head_ids], "lr": C["HEAD_LR"]},
                          {"params": [p for p in params if id(p) not in head_ids], "lr": C["LR"]}) if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"])
    total = math.ceil(len(train_ex) / C["BATCH_SIZE"]) * C["EPOCHS"]; warm = int(C["WARMUP_FRAC"] * total)
    base = [g["lr"] for g in opt.param_groups]
    amp = C["AMP"] and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp) if hasattr(torch.amp, "GradScaler") else torch.cuda.amp.GradScaler(enabled=amp)
    cw = class_weights(train_ex, model.head_out, C, device)
    rng = random.Random(C["SEED"]); best, best_state, hist, step, bad = -1.0, None, [], 0, 0
    t0 = time.time()
    for ep in range(1, C["EPOCHS"] + 1):
        model.train(); order = list(range(len(train_ex))); rng.shuffle(order); run = 0.0
        for bi in range(0, len(order), C["BATCH_SIZE"]):
            ids, att, seg, lab, S, pos = collate([train_ex[i] for i in order[bi:bi + C["BATCH_SIZE"]]], pad_id, device)
            for g, b in zip(opt.param_groups, base):
                g["lr"] = b * (step / max(1, warm) if step < warm else max(0.0, (total - step) / max(1, total - warm)))
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp):
                lg = model(ids, att, seg, S, pos).float()
                loss = F.cross_entropy(lg.reshape(-1, lg.size(-1)), lab.reshape(-1), weight=cw, ignore_index=-100,
                                       label_smoothing=C["LABEL_SMOOTHING"])
                if teacher is not None:
                    with torch.no_grad():
                        tl = teacher(ids, att, seg, S, pos).float()
                    m = lab.reshape(-1) != -100; T_ = C["KD_TEMPERATURE"]
                    kd = F.kl_div(F.log_softmax(lg.reshape(-1, lg.size(-1))[m] / T_, -1),
                                  F.softmax(tl.reshape(-1, tl.size(-1))[m] / T_, -1), reduction="batchmean") * T_ * T_
                    loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, C["GRAD_CLIP"]); scaler.step(opt); scaler.update()
            run += loss.item(); step += 1
            if step % C["LOG_EVERY"] == 0:
                log(f"  epoch {ep} step {step}/{total} loss {run / C['LOG_EVERY']:.4f} ({time.time() - t0:.0f}s)"); run = 0.0
        if dev_docs:
            res = evaluate_docs(predict_logits(model, dev_docs, C, pad_id, device), dev_docs, id2label, labels)
            hist.append({"epoch": ep, "dev_macro_F1": res["macro_F1"], **{k: v for k, v in res.items() if k.startswith("macro_F1_")}})
            log(f"epoch {ep}: dev macro-F1 {res['macro_F1']:.4f} | " + " ".join(f"{k[9:]} {v:.4f}" for k, v in res.items() if k.startswith('macro_F1_'))
                + f" | acc {res['accuracy']:.4f} ({time.time() - t0:.0f}s)")
            if res["macro_F1"] > best:
                best, bad = res["macro_F1"], 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= C["PATIENCE"]: log(f"early stop after {bad} epochs without improvement"); break
    if best_state is not None: model.load_state_dict(best_state)
    return {"best_dev_macro_F1": best, "history": hist, "train_time_s": time.time() - t0, "steps": step}

### 1. Data

In [ ]:
if SMOKE_TEST:
    labels = ["Fact", "Issue", "ArgumentPetitioner", "ArgumentRespondent", "Statute", "Dissent", "PrecedentReliedUpon",
              "PrecedentNotReliedUpon", "PrecedentOverruled", "RulingByLowerCourt", "RatioOfTheDecision", "RulingByPresentCourt", "None"]
    _S = [("The appellant filed a suit for recovery of money.", "Fact"), ("The question is whether section 9 applies.", "Issue"),
          ("Counsel for the appellant argued that the lower court erred.", "ArgumentPetitioner"),
          ("The respondent submitted that the claim is barred.", "ArgumentRespondent"),
          ("Section 9 of the Act reads as follows.", "Statute"), ("The High Court dismissed the petition.", "RulingByLowerCourt"),
          ("We are of the view that the provision is mandatory.", "RatioOfTheDecision"), ("The appeal is allowed.", "RulingByPresentCourt")]
    def _doc(i, dom, lab=True):
        sents = _S * 3
        return {"id": f"{dom}{i}", "text": [s for s, _ in sents], **({"labels": [l for _, l in sents]} if lab else {})}
    raw = {f"{D}_{sp}": [_doc(i, D, sp != "test") for i in range(3)] for D in ("CL", "IT") for sp in ("train", "dev", "test")}
    get = lambda D, sp: raw[f"{D}_{sp}"]
else:
    labels = load_label_vocab(load_json(data_file("Train/label_vocab.json")))
    get = lambda D, sp: load_json(data_file(f"{'Test' if sp == 'test' else 'Train'}/{D}_{sp}.json"))
print(f"labels ({len(labels)}): {labels}")
label2id = {l: i for i, l in enumerate(labels)}; id2label = dict(enumerate(labels))
train_docs = parse_docs(get("CL", "train"), "CL") + parse_docs(get("IT", "train"), "IT")
dev_docs = parse_docs(get("CL", "dev"), "CL") + parse_docs(get("IT", "dev"), "IT")
test_docs = parse_docs(get("CL", "test"), "CL") + parse_docs(get("IT", "test"), "IT")
for name, dd in (("train", train_docs), ("dev", dev_docs), ("test", test_docs)): check_docs(dd, labels, name)
_ids = [d["id"] for d in test_docs]; assert len(_ids) == len(set(_ids)), "test document ids repeat across CL/IT"
lab_counts = Counter(l for d in train_docs for l in d["labels"])
print("train label counts:", dict(lab_counts.most_common()))
for D in ("CL", "IT"):
    print(D, "train sentences:", sum(len(d["sents"]) for d in train_docs if d["domain"] == D),
          "| dev:", sum(len(d["sents"]) for d in dev_docs if d["domain"] == D))

if SMOKE_TEST:
    tokenizer = LegalBPETokenizer.train([" ".join(s for s, _ in _S)] * 20, vocab_size=400, verbose=False)
else:
    MODEL_PATH = find_model_dir(C["ARCH"]); print("tokenizer from", MODEL_PATH)
    tokenizer = LegalBPETokenizer.from_pretrained(MODEL_PATH)
for dd in (train_docs, dev_docs, test_docs): encode_docs(dd, tokenizer, C)
_lens = sorted(len(x) for d in train_docs for x in d["ids"])
print(f"sentence length (tokens): median {_lens[len(_lens)//2]}, 95th pct {_lens[int(len(_lens)*.95)]}, max {_lens[-1]} "
      f"(truncated to {MAX_SENT_TOKENS}: {sum(l >= MAX_SENT_TOKENS for l in _lens)} sentences)")
print("training windows (doc mode):", len(build_examples(train_docs, C, tokenizer.eos_token_id, label2id)))

### 2. Experiment runner

In [ ]:
import gc, traceback

def build_rr(c):
    arch, method = c["ARCH"], c["METHOD"]
    if SMOKE_TEST:
        cfg = dict(arch=arch, vocab_size=len(tokenizer), n_embd=32, n_layer=4, n_enc_layer=4, n_dec_layer=2, n_head=4,
                   ffn_hidden=64, rope_theta=1e4, dropout=0.0, max_positions=1344,
                   bos_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
        bb = ARCH_CLASSES[arch](cfg)
    else:
        cfg, bb = load_backbone(find_model_dir(arch), ARCH_CLASSES)
    if method == "kd":
        bb = keep_layers(arch, bb, c["KD_KEEP_LAYERS"] if not SMOKE_TEST else [0, 2])
    m = RRModel(arch, bb, len(labels), context=c["CONTEXT"], sent_head=c["SENT_HEAD"],
                n_prompt=c["N_PROMPT"] if method == "prompt" else 0,
                pos_buckets=c["POS_BUCKETS"] if c["USE_POSITION"] else 0)
    info = apply_method(m, method, c)
    return m.to(device), info

def load_teacher(c):
    tdir = os.path.join(WORK, c["KD_TEACHER_RUN"]) if c.get("KD_TEACHER_RUN") else c["KD_TEACHER_DIR"]
    assert tdir and os.path.exists(os.path.join(tdir, "rr_model.pt")), f"teacher weights not found in {tdir}"
    tc = json.load(open(os.path.join(tdir, "run_config.json")))
    t, tinfo = build_rr(tc)
    t.load_state_dict(torch.load(os.path.join(tdir, "rr_model.pt"), map_location="cpu")); t.eval()
    for p in t.parameters(): p.requires_grad = False
    print(f"teacher: {tc['RUN_NAME']} ({tinfo['total_params'] / 1e6:.1f}M params)")
    return t, tc["RUN_NAME"]

def run_ensemble(c):
    out = os.path.join(WORK, c["RUN_NAME"]); os.makedirs(out, exist_ok=True)
    runs = c["ENSEMBLE_RUNS"]
    if runs == "top3":
        ok = sorted([m for m in summary if m.get("status") == "ok" and m["method"] != "ensemble"], key=lambda m: -m["dev_macro_F1"])
        runs = [m["run"] for m in ok[:3]]
    parts = [torch.load(os.path.join(WORK, r, "logits.pt")) for r in runs]
    avg = lambda key: [torch.log(sum(torch.softmax(p[key][i].float(), -1) for p in parts) / len(parts)) for i in range(len(parts[0][key]))]
    dev_lg, test_lg = avg("dev"), avg("test")
    res = evaluate_docs(dev_lg, dev_docs, id2label, labels)
    print(f"ENSEMBLE of {runs}: dev macro-F1 {res['macro_F1']:.4f} | " + " ".join(f"{k[9:]} {v:.4f}" for k, v in res.items() if k.startswith("macro_F1_")))
    pred = {d["id"]: [id2label[int(i)] for i in lg.argmax(-1)] for d, lg in zip(test_docs, test_lg)}
    for d in test_docs: assert len(pred[d["id"]]) == len(d["sents"])
    json.dump(pred, open(os.path.join(out, "prediction.json"), "w"))
    with zipfile.ZipFile(os.path.join(out, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(out, "prediction.json"), "prediction.json")
    infos = [json.load(open(os.path.join(WORK, r, "metrics.json"))) for r in runs]
    metrics = {"run": c["RUN_NAME"], "arch": "ensemble", "method": "ensemble", "members": runs, "teacher": None,
               "dev_macro_F1": res["macro_F1"], "dev_macro_P": res["macro_P"], "dev_macro_R": res["macro_R"],
               "dev_accuracy": res["accuracy"], **{k: v for k, v in res.items() if k.startswith("macro_F1_")},
               "per_label": res["per_label"], "confusions": res["confusions"],
               "trainable_params": sum(m["trainable_params"] for m in infos), "total_params": sum(m["total_params"] for m in infos),
               "train_time_s": sum(m["train_time_s"] for m in infos), "peak_gpu_GB": 0.0, "history": []}
    json.dump(metrics, open(os.path.join(out, "metrics.json"), "w"), indent=2)
    return metrics

def run_experiment(c):
    if c["METHOD"] == "ensemble":
        return run_ensemble(c)
    random.seed(c["SEED"]); np.random.seed(c["SEED"]); torch.manual_seed(c["SEED"])
    out = os.path.join(WORK, c["RUN_NAME"]); os.makedirs(out, exist_ok=True)
    pad = tokenizer.eos_token_id
    model, info = build_rr(c)
    print(f"{c['ARCH']} + {c['METHOD']} (context {c['CONTEXT_MODE']}, lr {c['LR']}): trainable "
          f"{info['trainable_params'] / 1e6:.2f}M / {info['total_params'] / 1e6:.2f}M ({info['trainable_pct']:.2f}%)")
    teacher, tname = load_teacher(c) if c["METHOD"] == "kd" else (None, None)
    train_ex = build_examples(train_docs, c, pad, label2id)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    result = train(model, train_ex, dev_docs, c, pad, device, id2label, labels, teacher=teacher)
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0.0
    del teacher
    t0 = time.time(); dev_lg = predict_logits(model, dev_docs, c, pad, device); dev_s = time.time() - t0
    res = evaluate_docs(dev_lg, dev_docs, id2label, labels)
    print(f"\nDEV {c['RUN_NAME']}: macro-F1 {res['macro_F1']:.4f} (P {res['macro_P']:.4f}, R {res['macro_R']:.4f}) | "
          + " ".join(f"{k[9:]} {v:.4f}" for k, v in res.items() if k.startswith("macro_F1_")) + f" | acc {res['accuracy']:.4f}")
    for l, v in sorted(res["per_label"].items(), key=lambda x: -x[1]["F1"]):
        print(f"  {l:24s} P {v['P']:.3f} R {v['R']:.3f} F1 {v['F1']:.3f} (n={v['support']})")
    print("  most common confusions (gold->pred):", res["confusions"])
    t0 = time.time(); test_lg = predict_logits(model, test_docs, c, pad, device); test_s = time.time() - t0
    pred = {d["id"]: [id2label[int(i)] for i in lg.argmax(-1)] for d, lg in zip(test_docs, test_lg)}
    for d in test_docs:
        assert len(pred[d["id"]]) == len(d["sents"]) and all(p in label2id for p in pred[d["id"]])
    json.dump(pred, open(os.path.join(out, "prediction.json"), "w"))
    with zipfile.ZipFile(os.path.join(out, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(out, "prediction.json"), "prediction.json")
    for D in ("CL", "IT"):
        json.dump({d["id"]: pred[d["id"]] for d in test_docs if d["domain"] == D}, open(os.path.join(out, f"prediction_{D}.json"), "w"))
    json.dump({d["id"]: [id2label[int(i)] for i in lg.argmax(-1)] for d, lg in zip(dev_docs, dev_lg)},
              open(os.path.join(out, "dev_predictions.json"), "w"))
    torch.save({"dev": [x.half() for x in dev_lg], "test": [x.half() for x in test_lg]}, os.path.join(out, "logits.pt"))
    json.dump({k: v for k, v in c.items() if isinstance(v, (int, float, str, bool, list, type(None)))},
              open(os.path.join(out, "run_config.json"), "w"), indent=2)
    metrics = {"run": c["RUN_NAME"], "arch": c["ARCH"], "method": c["METHOD"], "context_mode": c["CONTEXT_MODE"],
               "sent_head": c["SENT_HEAD"], "lr": c["LR"], "teacher": tname,
               "dev_macro_F1": res["macro_F1"], "dev_macro_P": res["macro_P"], "dev_macro_R": res["macro_R"],
               "dev_accuracy": res["accuracy"], **{k: v for k, v in res.items() if k.startswith("macro_F1_")},
               "per_label": res["per_label"], "confusions": res["confusions"], **info,
               "train_time_s": result["train_time_s"], "epochs_run": len(result["history"]), "history": result["history"],
               "peak_gpu_GB": peak, "dev_inference_s": dev_s, "test_inference_s": test_s}
    json.dump(metrics, open(os.path.join(out, "metrics.json"), "w"), indent=2)
    if c["METHOD"] in ("full", "partial") or c["SAVE_WEIGHTS"] or any(e.get("KD_TEACHER_RUN") == c["RUN_NAME"] for e in EXPERIMENTS):
        torch.save(model.state_dict(), os.path.join(out, "rr_model.pt"))
    print(f"saved {out} | train {result['train_time_s'] / 60:.1f} min | peak GPU {peak:.2f} GB")
    del model; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return metrics

### 3. Run all experiments

In [ ]:
run_start, summary = time.time(), []
for i, e in enumerate(EXPERIMENTS, 1):
    c = experiment_config(e)
    used = (time.time() - run_start) / 3600
    longest = max([m["train_time_s"] for m in summary if "train_time_s" in m] or [0]) / 3600
    if used + 1.2 * longest > TIME_BUDGET_HOURS:
        print(f"SKIP {c['RUN_NAME']}: time budget"); summary.append({"run": c["RUN_NAME"], "status": "skipped"}); continue
    print("\n" + "=" * 100 + f"\nEXPERIMENT {i}/{len(EXPERIMENTS)}: {c['RUN_NAME']}   ({used:.2f} h elapsed)\n" + "=" * 100)
    try:
        summary.append(dict(run_experiment(c), status="ok"))
    except Exception as ex:
        traceback.print_exc(); summary.append({"run": c["RUN_NAME"], "status": f"FAILED: {type(ex).__name__}: {ex}"})
        gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()
print("\n" + "=" * 100 + "\nSUMMARY (dev = official CL_dev + IT_dev)")
print(f"{'run':34s} {'macroF1':>7s} {'CL':>6s} {'IT':>6s} {'acc':>6s} {'trainable':>10s} {'total':>7s} {'min':>5s} {'GB':>5s}")
for m in summary:
    if m.get("status") != "ok": print(f"{m['run']:34s} {m['status']}"); continue
    print(f"{m['run']:34s} {m['dev_macro_F1']:7.4f} {m.get('macro_F1_CL', 0):6.3f} {m.get('macro_F1_IT', 0):6.3f} "
          f"{m['dev_accuracy']:6.3f} {m['trainable_params'] / 1e6:9.2f}M {m['total_params'] / 1e6:6.1f}M "
          f"{m['train_time_s'] / 60:5.1f} {m['peak_gpu_GB']:5.2f}")
json.dump(summary, open(os.path.join(WORK, "phase3_summary.json"), "w"), indent=2)
with zipfile.ZipFile(os.path.join(WORK, "phase3_all_submissions.zip"), "w", zipfile.ZIP_DEFLATED) as z:
    z.write(os.path.join(WORK, "phase3_summary.json"), "phase3_summary.json")
    for i, m in enumerate(summary, 1):
        d = os.path.join(WORK, m["run"])
        if os.path.exists(os.path.join(d, "submission.zip")):
            z.write(os.path.join(d, "submission.zip"), f"codabench/{i}_{m['run']}.zip")
        for f in ("submission.zip", "metrics.json", "run_config.json", "dev_predictions.json", "prediction_CL.json", "prediction_IT.json"):
            if os.path.exists(os.path.join(d, f)): z.write(os.path.join(d, f), f"{m['run']}/{f}")
print("Download phase3_all_submissions.zip; upload the files in its codabench/ folder (named <n>_<run>.zip) to Codabench.")